# Test03 oncology genes: grab recent literature

Use PubTator3 gene annotations to discover PMIDs for a broad oncology gene panel, then fetch PubMed/PMC article sections and save section-level literature blocks for pretagging.

In [1]:
from __future__ import annotations

import http.client
import os
import urllib.error
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from tqdm.notebook import tqdm

from dgilit import PubMedClient

load_dotenv()

MC_EMAIL = os.environ["MC_EMAIL"]
PUBTATOR_GENE_PATH = Path(
    os.environ.get("PUBTATOR_GENE_PATH")
    or os.environ.get("PUBTATOR3_GENE_PATH")
    or os.environ.get("GENE2PUBTATOR3_PATH", "")
)

if not PUBTATOR_GENE_PATH.exists():
    raise FileNotFoundError(
        "Set PUBTATOR_GENE_PATH to the PubTator3 gene annotation file "
        "before running this notebook."
    )

OUTPUT_DIR = Path("../../../data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LITERATURE_XLSX = OUTPUT_DIR / "2026-07-02-test03-oncology-genes-literature.xlsx"
LITERATURE_PARQUET = OUTPUT_DIR / "2026-07-02-test03-oncology-genes-literature.parquet"
GENE_PMID_XLSX = OUTPUT_DIR / "2026-07-02-test03-oncology-genes-pubtator-pmids.xlsx"
RECENT_PMID_XLSX = OUTPUT_DIR / "2026-07-02-test03-oncology-genes-recent-pmids.xlsx"
FETCH_ERRORS_CSV = OUTPUT_DIR / "2026-07-02-test03-oncology-genes-fetch-errors.csv"

MIN_PUBLICATION_YEAR = 2021
MAX_PMIDS = None  # Set to an int for a quick smoke test before running the full experiment.

GENE_SYMBOLS = [
    "BCL2", "EGFR", "ALK", "ROS1", "BRAF", "KRAS", "MET", "ERBB2", "RET",
    "NTRK1", "NTRK2", "NTRK3", "ESR1", "AR", "PIK3CA", "PTEN", "BRCA1",
    "BRCA2", "PARP1", "CDK4", "CDK6", "MTOR", "FGFR1", "FGFR2", "FGFR3",
    "IDH1", "IDH2", "FLT3", "KIT",
]

GENE_ALIASES = {
    "ERBB2": {"ERBB2", "HER2"},
    "MTOR": {"MTOR", "mTOR"},
}

for symbol in GENE_SYMBOLS:
    GENE_ALIASES.setdefault(symbol, {symbol})

alias_to_symbol = {
    alias.upper(): symbol
    for symbol, aliases in GENE_ALIASES.items()
    for alias in aliases
}

len(GENE_SYMBOLS), PUBTATOR_GENE_PATH

(29,
 PosixPath('/Users/mjc014/Documents/repo/git/dgilits/BACKUPS DONT DLEETE/gene2pubtator3'))

In [2]:
PUBTATOR_GENE_COLUMNS = [
    "pmid",
    "gene_name",
    "gene_mention",
    "gene_concept_id",
    "mention_start",
    "mention_end",
]

GENE_ENTITY_TYPES = {"gene", "genes", "geneorgeneproduct", "g"}
GENE_ID_TO_SYMBOL = globals().get("gene_id_to_symbol", {})


def split_aliases(value: str | None) -> list[str]:
    if value is None:
        return []
    return [alias.strip() for alias in str(value).split("|") if alias.strip()]


def infer_pubtator_match(mention_or_aliases: str, concept_id: str | None) -> tuple[str | None, str | None]:
    for alias in split_aliases(mention_or_aliases):
        symbol = alias_to_symbol.get(alias.upper())
        if symbol:
            return symbol, alias

    if concept_id:
        for value in str(concept_id).replace(";", ",").split(","):
            value = value.strip()
            if not value:
                continue

            symbol = alias_to_symbol.get(value.upper())
            if symbol:
                return symbol, value

            symbol = GENE_ID_TO_SYMBOL.get(value)
            if symbol:
                return symbol, value

    return None, None


def parse_pubtator_gene_row(parts: list[str]) -> dict | None:
    """Parse supported PubTator3 gene row shapes.

    Supported shapes:
    - PMID<TAB>Gene<TAB>GeneID<TAB>Aliases<TAB>PubTator3
    - PMID<TAB>start<TAB>end<TAB>mention<TAB>Gene<TAB>identifier
    """
    if len(parts) >= 5 and parts[1].strip().lower() in GENE_ENTITY_TYPES:
        pmid, entity_type, concept_id, aliases, source = parts[:5]
        symbol, matched_alias = infer_pubtator_match(aliases, concept_id)
        if not symbol:
            return None
        return {
            "pmid": str(pmid),
            "gene_name": symbol,
            "gene_mention": matched_alias or aliases,
            "gene_concept_id": concept_id or None,
            "mention_start": None,
            "mention_end": None,
        }

    if len(parts) >= 6 and parts[4].strip().lower() in GENE_ENTITY_TYPES:
        pmid, start, end, mention, entity_type, concept_id = parts[:6]
        symbol, matched_alias = infer_pubtator_match(mention, concept_id)
        if not symbol:
            return None
        return {
            "pmid": str(pmid),
            "gene_name": symbol,
            "gene_mention": matched_alias or mention,
            "gene_concept_id": concept_id or None,
            "mention_start": int(start),
            "mention_end": int(end),
        }

    return None


def scan_pubtator_gene_pmids(path: Path, alias_to_symbol: dict[str, str]) -> pd.DataFrame:
    """Return PubTator3 gene annotation rows matching this experiment's gene panel."""
    rows = []
    entity_type_counts = Counter()
    candidate_gene_rows = 0
    sampled_gene_rows = []
    sampled_nonmatching_gene_rows = []

    with path.open() as handle:
        for line in tqdm(handle, desc="Scanning PubTator3 gene annotations"):
            line = line.rstrip("\n")
            if not line or "|t|" in line or "|a|" in line:
                continue

            parts = line.split("\t")
            if len(parts) < 5:
                continue

            entity_type = None
            if parts[1].strip().lower() in GENE_ENTITY_TYPES:
                entity_type = parts[1]
            elif len(parts) >= 6 and parts[4].strip().lower() in GENE_ENTITY_TYPES:
                entity_type = parts[4]

            if entity_type is None:
                if len(parts) > 1:
                    entity_type_counts[parts[1].strip().lower()] += 1
                continue

            normalized_entity_type = entity_type.strip().lower()
            entity_type_counts[normalized_entity_type] += 1
            candidate_gene_rows += 1

            if len(sampled_gene_rows) < 5:
                sampled_gene_rows.append(parts[:6])

            parsed = parse_pubtator_gene_row(parts)
            if parsed is None:
                if len(sampled_nonmatching_gene_rows) < 5:
                    sampled_nonmatching_gene_rows.append(parts[:6])
                continue

            rows.append(parsed)

    print("Most common PubTator entity types:")
    print(entity_type_counts.most_common(10))
    print(f"Gene-like annotation rows scanned: {candidate_gene_rows}")

    if not rows:
        print("No matching panel genes were found. Sample gene-like rows:")
        for sample in sampled_gene_rows:
            print(sample)
        return pd.DataFrame(columns=PUBTATOR_GENE_COLUMNS)

    if sampled_nonmatching_gene_rows:
        print("Sample gene-like rows that did not match the panel:")
        for sample in sampled_nonmatching_gene_rows:
            print(sample)

    return (
        pd.DataFrame(rows, columns=PUBTATOR_GENE_COLUMNS)
        .drop_duplicates(subset=["pmid", "gene_name", "gene_mention", "gene_concept_id"])
        .reset_index(drop=True)
    )


gene_pmid_df = scan_pubtator_gene_pmids(PUBTATOR_GENE_PATH, alias_to_symbol)

print(f"Found {len(gene_pmid_df)} PubTator3 gene annotation rows for panel genes")
print(f"Found {gene_pmid_df['pmid'].nunique()} unique PMIDs before date filtering")

gene_pmid_df.head()


Scanning PubTator3 gene annotations: 0it [00:00, ?it/s]

Most common PubTator entity types:
[('gene', 72670276)]
Gene-like annotation rows scanned: 72670276
Sample gene-like rows that did not match the panel:
['40591000', 'Gene', '1084', 'CEA|carcinoembryonic antigen', 'PubTator3']
['40591000', 'Gene', '94025', 'carbohydrate antigen 125|CA125', 'PubTator3']
['40486000', 'Gene', '3630', 'C-peptide', 'PubTator3']
['40510000', 'Gene', '1401', 'C-reactive protein', 'PubTator3']
['40510000', 'Gene', '3569', 'interleukin-6', 'PubTator3']
Found 1298859 PubTator3 gene annotation rows for panel genes
Found 691424 unique PMIDs before date filtering


,pmid,gene_name,gene_mention,gene_concept_id,mention_start,mention_end
0,40498000,NTRK1,Ntrk1,18211,None,None
1,40224001,EGFR,EGFR,1956,None,None
2,40224001,ERBB2,HER2,2064,None,None
3,40480001,EGFR,EGFR,13649,None,None
4,40401001,AR,AR,367,None,None


In [3]:
def join_unique(values: pd.Series) -> str:
    return "; ".join(sorted({str(value) for value in values.dropna() if str(value)}))


FETCH_ERRORS = []
FETCH_ERROR_TYPES = (
    ET.ParseError,
    http.client.IncompleteRead,
    TimeoutError,
    urllib.error.HTTPError,
    urllib.error.URLError,
)


def record_fetch_error(stage: str, pmids: list[str], error: Exception) -> None:
    FETCH_ERRORS.append({
        "stage": stage,
        "pmids": ";".join(map(str, pmids)),
        "pmid_count": len(pmids),
        "error_type": type(error).__name__,
        "error_message": str(error),
    })


def fetch_articles_with_progress(
    client: PubMedClient,
    pmids: list[str],
    *,
    chunk_size: int = 100,
    include_full_text: bool = True,
    desc: str = "Fetching PubMed/PMC article chunks",
):
    articles = {}
    stage = "full_text" if include_full_text else "metadata"

    for start in tqdm(
        range(0, len(pmids), chunk_size),
        total=(len(pmids) + chunk_size - 1) // chunk_size,
        desc=desc,
    ):
        chunk = pmids[start:start + chunk_size]
        try:
            articles.update(
                client.fetch_articles(
                    chunk,
                    include_full_text=include_full_text,
                )
            )
            continue
        except FETCH_ERROR_TYPES as error:
            record_fetch_error(stage=f"{stage}_chunk", pmids=chunk, error=error)
        except Exception as error:
            record_fetch_error(stage=f"{stage}_chunk_unexpected", pmids=chunk, error=error)

        if len(chunk) == 1:
            continue

        for pmid in tqdm(chunk, desc=f"Retrying failed {stage} chunk one PMID at a time", leave=False):
            try:
                articles.update(
                    client.fetch_articles(
                        [pmid],
                        include_full_text=include_full_text,
                    )
                )
            except FETCH_ERROR_TYPES as error:
                record_fetch_error(stage=f"{stage}_pmid", pmids=[pmid], error=error)
            except Exception as error:
                record_fetch_error(stage=f"{stage}_pmid_unexpected", pmids=[pmid], error=error)

    return articles


def publication_year_as_int(article) -> int | None:
    if not article.publication_year:
        return None
    try:
        return int(article.publication_year)
    except ValueError:
        return None


if gene_pmid_df.empty:
    raise ValueError(
        "No PubTator3 PMIDs matched the oncology gene panel. "
        "Check PUBTATOR_GENE_PATH and inspect the sample rows printed above."
    )

article_gene_hits = (
    gene_pmid_df
    .groupby("pmid", as_index=False)
    .agg(
        gene_names=("gene_name", join_unique),
        gene_mentions=("gene_mention", join_unique),
        gene_concept_ids=("gene_concept_id", join_unique),
    )
)

pmids = article_gene_hits["pmid"].astype(str).tolist()
if MAX_PMIDS is not None:
    pmids = pmids[:MAX_PMIDS]

print(f"Preparing to precheck {len(pmids)} unique PMIDs")

client = PubMedClient(
    email=MC_EMAIL,
    batch_size=100,
)

metadata_articles = fetch_articles_with_progress(
    client,
    pmids,
    chunk_size=100,
    include_full_text=False,
    desc="Prechecking PubMed metadata chunks",
)

recent_pmid_rows = []
for pmid, article in tqdm(
    metadata_articles.items(),
    total=len(metadata_articles),
    desc="Filtering PMIDs by publication year",
):
    publication_year = publication_year_as_int(article)
    if publication_year is None or publication_year < MIN_PUBLICATION_YEAR:
        continue
    recent_pmid_rows.append({
        "pmid": str(pmid),
        "publication_year": publication_year,
        "title": article.title,
        "journal": article.journal,
    })

recent_pmid_df = pd.DataFrame(recent_pmid_rows)
recent_pmids = recent_pmid_df["pmid"].astype(str).tolist() if not recent_pmid_df.empty else []

recent_pmid_df.to_excel(RECENT_PMID_XLSX, index=False)

print(f"Metadata fetched for {len(metadata_articles)} PMIDs")
print(f"Kept {len(recent_pmids)} PMIDs published since {MIN_PUBLICATION_YEAR}")
print(f"Fetch errors so far: {len(FETCH_ERRORS)}")
print(f"Saved recent PMID checkpoint to {RECENT_PMID_XLSX}")

client = PubMedClient(
    email=MC_EMAIL,
    batch_size=25,
)

articles = fetch_articles_with_progress(
    client,
    recent_pmids,
    chunk_size=25,
    include_full_text=True,
    desc="Fetching recent PubMed/PMC article chunks",
)

fetch_errors_df = pd.DataFrame(FETCH_ERRORS)
if not fetch_errors_df.empty:
    fetch_errors_df.to_csv(FETCH_ERRORS_CSV, index=False)
    print(f"Saved {len(fetch_errors_df)} fetch errors to {FETCH_ERRORS_CSV}")
else:
    print("No fetch errors recorded")

print(f"Fetched {len(articles)} recent PubMed article records")
print(f"PMC full text available for {sum(article.full_text_available for article in articles.values())} records")


Preparing to precheck 691424 unique PMIDs


Prechecking PubMed metadata chunks:   0%|          | 0/6915 [00:00<?, ?it/s]

Retrying failed metadata chunk one PMID at a time:   0%|          | 0/100 [00:00<?, ?it/s]

Retrying failed metadata chunk one PMID at a time:   0%|          | 0/100 [00:00<?, ?it/s]

Retrying failed metadata chunk one PMID at a time:   0%|          | 0/100 [00:00<?, ?it/s]

...

Retrying failed full_text chunk one PMID at a time:   0%|          | 0/25 [00:00<?, ?it/s]

Retrying failed full_text chunk one PMID at a time:   0%|          | 0/25 [00:00<?, ?it/s]

Retrying failed full_text chunk one PMID at a time:   0%|          | 0/25 [00:00<?, ?it/s]

Retrying failed full_text chunk one PMID at a time:   0%|          | 0/25 [00:00<?, ?it/s]

Saved 18 fetch errors to ../../../data/2026-07-02-test03-oncology-genes-fetch-errors.csv
Fetched 260510 recent PubMed article records
PMC full text available for 223126 records


In [ ]:
gene_hits_by_pmid = article_gene_hits.set_index("pmid").to_dict(orient="index")

section_rows = []

for pmid, article in tqdm(
    articles.items(),
    total=len(articles),
    desc="Building section-level literature blocks",
):
    publication_year = publication_year_as_int(article)
    if publication_year is None or publication_year < MIN_PUBLICATION_YEAR:
        continue

    gene_hits = gene_hits_by_pmid.get(str(pmid), {})

    for section_idx, section in enumerate(article.sections):
        if not section.text:
            continue

        section_rows.append({
            "pmid": str(pmid),
            "pmcid": article.pmcid,
            "publication_year": publication_year,
            "title": article.title,
            "journal": article.journal,
            "full_text_available": article.full_text_available,
            "block_id": f"{pmid}:{section_idx}",
            "section_index": section_idx,
            "section_type": section.section_type,
            "section_label": section.label,
            "section_id": section.section_id,
            "parent_label": section.parent_label,
            "section_source": section.source,
            "context": section.text,
            "gene_names": gene_hits.get("gene_names"),
            "gene_mentions": gene_hits.get("gene_mentions"),
            "gene_concept_ids": gene_hits.get("gene_concept_ids"),
            "literature_source": "pubtator3gene",
        })

literature_df = pd.DataFrame(section_rows)

print(f"Kept {literature_df['pmid'].nunique() if not literature_df.empty else 0} PMIDs published since {MIN_PUBLICATION_YEAR}")
print(f"Created {len(literature_df)} section-level literature blocks")

literature_df.head()


Building section-level literature blocks:   0%|          | 0/260510 [00:00<?, ?it/s]

Kept 256871 PMIDs published since 2021
Created 4303213 section-level literature blocks


,pmid,pmcid,publication_year,title,journal,full_text_available,block_id,section_index,section_type,section_label,section_id,parent_label,section_source,context,gene_names,gene_mentions,gene_concept_ids,literature_source
0,29799308,PMC8054522,2021,Genetic Influences on Patient-Oriented Outcome...,Journal of neurotrauma,True,29799308:0,0,abstract,None,None,None,pmc,There is a growing literature on the impact of...,BCL2,BCL2,596,pubtator3gene
1,29799308,PMC8054522,2021,Genetic Influences on Patient-Oriented Outcome...,Journal of neurotrauma,True,29799308:1,1,body,Introduction,s001,None,pmc,Outcome prediction in severe traumatic brain i...,BCL2,BCL2,596,pubtator3gene
2,29799308,PMC8054522,2021,Genetic Influences on Patient-Oriented Outcome...,Journal of neurotrauma,True,29799308:2,2,body,Methods,s002,None,pmc,This review was conducted and reported in line...,BCL2,BCL2,596,pubtator3gene
3,29799308,PMC8054522,2021,Genetic Influences on Patient-Oriented Outcome...,Journal of neurotrauma,True,29799308:3,3,body,Inclusion/exclusion criteria,s003,Methods,pmc,We included all studies of five or more adult ...,BCL2,BCL2,596,pubtator3gene
4,29799308,PMC8054522,2021,Genetic Influences on Patient-Oriented Outcome...,Journal of neurotrauma,True,29799308:4,4,body,Search strategy,s004,Methods,pmc,"At the beginning of August 2017, EMBASE, MEDLI...",BCL2,BCL2,596,pubtator3gene


: 

In [ ]:
EXCEL_MAX_DATA_ROWS = 1_048_575  # leave room for header row

GENE_PMID_CSV = GENE_PMID_XLSX.with_suffix(".csv")
GENE_PMID_CSV_GZ = GENE_PMID_XLSX.with_suffix(".csv.gz")

LITERATURE_CSV = LITERATURE_XLSX.with_suffix(".csv")
LITERATURE_CSV_GZ = LITERATURE_XLSX.with_suffix(".csv.gz")

def write_excel_split(df, path, base_sheet_name="Sheet"):
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        for i, start in enumerate(range(0, len(df), EXCEL_MAX_DATA_ROWS), start=1):
            df.iloc[start:start + EXCEL_MAX_DATA_ROWS].to_excel(
                writer,
                sheet_name=f"{base_sheet_name}_{i}",
                index=False,
            )

save_steps = [
    ("PubTator gene PMID rows Excel", lambda: write_excel_split(gene_pmid_df, GENE_PMID_XLSX, "gene_pmids")),
    ("PubTator gene PMID rows CSV", lambda: gene_pmid_df.to_csv(GENE_PMID_CSV, index=False)),
    ("PubTator gene PMID rows CSV.GZ", lambda: gene_pmid_df.to_csv(GENE_PMID_CSV_GZ, index=False, compression="gzip")),

    ("Recent PMID checkpoint", lambda: recent_pmid_df.to_excel(RECENT_PMID_XLSX, index=False)),
    ("Fetch error log", lambda: pd.DataFrame(FETCH_ERRORS).to_csv(FETCH_ERRORS_CSV, index=False)),

    ("Literature Excel", lambda: write_excel_split(literature_df, LITERATURE_XLSX, "literature")),
    ("Literature CSV", lambda: literature_df.to_csv(LITERATURE_CSV, index=False)),
    ("Literature CSV.GZ", lambda: literature_df.to_csv(LITERATURE_CSV_GZ, index=False, compression="gzip")),
]

for label, save_fn in tqdm(save_steps, desc="Writing literature outputs"):
    tqdm.write(f"Writing {label}")
    save_fn()

try:
    with tqdm(total=1, desc="Writing literature parquet") as progress:
        literature_df.to_parquet(LITERATURE_PARQUET, index=False)
        progress.update(1)
except ImportError as exc:
    print(f"Skipping parquet export because an optional parquet engine is unavailable: {exc}")

for path in [
    GENE_PMID_XLSX,
    GENE_PMID_CSV,
    GENE_PMID_CSV_GZ,
    RECENT_PMID_XLSX,
    FETCH_ERRORS_CSV,
    LITERATURE_XLSX,
    LITERATURE_CSV,
    LITERATURE_CSV_GZ,
    LITERATURE_PARQUET,
]:
    print(path)

Writing literature outputs:   0%|          | 0/8 [00:00<?, ?it/s]

Writing PubTator gene PMID rows Excel
Writing PubTator gene PMID rows CSV
Writing PubTator gene PMID rows CSV.GZ
Writing Recent PMID checkpoint
Writing Fetch error log
Writing Literature Excel
